# Carbon Crunch — OCR Receipt Extraction Demo

**Author:** Tirupathamma Guntru

End-to-end demo of the OCR pipeline: preprocessing → text recognition →
field extraction → confidence scoring → financial summary.

## Pipeline
```
Receipt image → Preprocess (denoise + contrast + deskew) → EasyOCR
             → Regex/keyword extractors → Confidence aggregation → JSON
```

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd()))

from src.preprocess import preprocess, estimate_blur, estimate_brightness, to_grayscale
from src.ocr import run_ocr, get_full_text
from src.extractor import extract_date, extract_store_name, extract_total, extract_items
from src.confidence import adjust_confidence, collect_low_confidence_flags
from src.summary import generate_summary
from main import process_receipt, run_pipeline

DATA_DIR = Path('data/receipts')
OUT_DIR  = Path('outputs')
print('Receipts in folder:', len(list(DATA_DIR.glob('*'))))

## 2. Pick a sample receipt and inspect it

In [ ]:
samples = sorted(p for p in DATA_DIR.iterdir()
                 if p.suffix.lower() in ('.jpg','.jpeg','.png'))
if not samples:
    raise FileNotFoundError('No receipt images found in data/receipts/. '
                            'Download the dataset and place images there.')

sample = samples[0]
print('Sample:', sample.name)

original = cv2.imread(str(sample))
plt.figure(figsize=(7,9))
plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
plt.title(f'Original — {sample.name}')
plt.axis('off')
plt.show()

## 3. Preprocessing — before vs after

In [ ]:
processed, metrics = preprocess(sample, return_metrics=True)
print('Quality metrics:', metrics)

fig, axs = plt.subplots(1, 2, figsize=(14,9))
axs[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axs[0].set_title('Original'); axs[0].axis('off')
axs[1].imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
axs[1].set_title('After Preprocess'); axs[1].axis('off')
plt.tight_layout(); plt.show()

## 4. OCR — detected lines with confidence

In [ ]:
lines = run_ocr(processed)
print(f'Detected {len(lines)} lines\n')

df = pd.DataFrame([{
    'text': l['text'],
    'confidence': round(l['confidence'], 3),
    'y': round(l['y_center'], 1),
} for l in lines])
df.head(30)

In [ ]:
# Visualize OCR confidence distribution
df['confidence'].hist(bins=20, edgecolor='black')
plt.title('Per-line OCR Confidence')
plt.xlabel('Confidence'); plt.ylabel('# lines')
plt.show()

## 5. Field extraction

In [ ]:
result = process_receipt(sample)
print(json.dumps({k: v for k, v in result.items() if k != 'raw_ocr_text'},
                 indent=2, ensure_ascii=False))

## 6. Run full pipeline on all receipts

In [ ]:
results = run_pipeline(DATA_DIR, OUT_DIR, save_previews=True, max_previews=5)
print(f'\nProcessed {len(results)} receipts.')

## 7. Confidence dashboard across all receipts

In [ ]:
rows = []
for r in results:
    if 'error' in r:
        continue
    rows.append({
        'file':        r['file'],
        'store_conf':  r['store_name']['confidence'],
        'date_conf':   r['date']['confidence'],
        'total_conf':  r['total_amount']['confidence'],
        'avg_ocr':     r['metadata']['average_ocr_confidence'],
        'low_flags':   ','.join(r['low_confidence_flags']) or '',
    })
conf_df = pd.DataFrame(rows)
conf_df.head(20)

In [ ]:
if not conf_df.empty:
    means = conf_df[['store_conf','date_conf','total_conf','avg_ocr']].mean()
    means.plot(kind='bar', color=['#6366f1','#10b981','#f59e0b','#94a3b8'])
    plt.title('Mean Confidence by Field (across all receipts)')
    plt.ylabel('Confidence (0-1)')
    plt.ylim(0, 1)
    plt.axhline(0.7, color='red', linestyle='--', label='Low-confidence threshold')
    plt.legend()
    plt.tight_layout(); plt.show()

## 8. Financial summary

In [ ]:
with open(OUT_DIR / 'summary.json') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

In [ ]:
stores = summary.get('spend_per_store', {})
if stores:
    sdf = pd.DataFrame([
        {'store': s, 'total': v['total'], 'transactions': v['transactions']}
        for s, v in stores.items()
    ]).head(10)
    sdf.plot(x='store', y='total', kind='barh', color='#6366f1', legend=False)
    plt.title('Top 10 Stores by Total Spend')
    plt.xlabel('Total Spend'); plt.gca().invert_yaxis()
    plt.tight_layout(); plt.show()
    sdf

## 9. Edge case demonstration

In [ ]:
# Show low-confidence cases — these are the ones flagged for human review
low_conf = [r for r in results
            if 'error' not in r and r.get('low_confidence_flags')]
print(f'Receipts flagged for review: {len(low_conf)}')
for r in low_conf[:5]:
    print(f"  {r['file']:40s}  flags={r['low_confidence_flags']}")

---
## Summary

✅ Preprocessing handles noise, blur, contrast, and skew

✅ EasyOCR extracts text with per-line confidence

✅ Regex + keyword heuristics extract structured fields

✅ Multi-signal confidence (OCR × pattern validation × heuristic)

✅ Low-confidence fields auto-flagged for human review

✅ Financial summary aggregates across all receipts